# 09 - Volatilite stochastique par signature : Heston et modele a retard

Suite du notebook 08 (Sec. 5.1 : GBM + payoff polynomial). On traite ici les deux
sections suivantes d'Abi Jaber & Gerard, ou la volatilite est **stochastique** et
represente elle-meme par une signature :

- **Sec. 5.2** -- call europeen ATM sous **Heston**, volatilite CIR (Tableau 6).
- **Sec. 5.3** -- call **asiatique geometrique** sous **volatilite a retard** (Tableau 7).

Quatre strategies sont comparees a chaque fois, plus une reference :

| | ce qui est appris | ce qui est utilise |
|---|---|---|
| **Linear BCS** | P&L empirique, trajectoire par trajectoire | signature lead-lag de S |
| **Linear REG** | forme quadratique shuffle (eq. 3.8-3.9) | **seulement** la signature esperee `E[S^LL]` |
| **Fourier BCS** | rien : sigma exact en forme close | Riccati + signature de `W_hat = (t, W)` |
| **Fourier REG** | sigma calibre sur les moments (Algorithme 2) | idem, avec le sigma appris |

Le notebook ne fait qu'importer et appeler les packages `models`, `signatures`,
`linear_hedging` et `fourier` : toute la mathematique est dans les docstrings des
modules.

> **Duree** : plusieurs minutes. Les postes couteux sont `E[S^LL]` a l'ordre 8
> (~9840 colonnes), les deltas Fourier (32 noeuds de Laguerre x 126 pas de Riccati)
> et les deux calibrations (gradient numerique sur 30 parametres).

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np

from models import (
    build_W_hat, build_Y_hat, compute_payoff_asian, compute_payoff_europeen,
    heston_variance_brownian, simulate_DE, simulate_heston, simulate_S_given_DE,
)
from signatures import (
    build_sigma_DE, calibrate_sigma, compute_sigma_CIR, dict_to_array, E_W_hat_all_times,
    generate_words, make_loss_fn, sigma_words, vec_to_sigma_dict, word_to_index,
)
from linear_hedging import (
    build_path_LL, build_reg_system, expected_signature, fit_xi_full_signature,
    fit_xi_power_words, hedging_word_indices, hedging_words, linear_bcs, pnl_stats,
    reg_pnl, sig_words, signature_transform, solve_reg, xi_r2,
)
from fourier import (
    compute_call_bs_asian, compute_delta_sig_fourier_asian_constant_rescaled,
    compute_delta_sig_fourier_constant_rescaled, compute_delta_vectorized,
    compute_naive_bs_asian_delta, compute_X0, compute_X0_sig_fourier_asian_constant_rescaled,
    compute_X0_sig_fourier_constant_rescaled, hedging_pnl, rescale_stream_signature, signed_floor,
)
from utils.plotting import plot_pnl_distribution

---
# Section 5.2 - Call europeen ATM sous Heston

$$dS_t = S_t\sqrt{V_t}\,dB_t,\qquad dV_t=\kappa(\theta-V_t)\,dt+\eta\sqrt{V_t}\,dW_t,\qquad d\langle B,W\rangle_t=\rho\,dt$$

avec $V_0=\theta=0.0625$, $\kappa=2$, $\eta=0.5$, $\rho=-0.7$, $T=1/2$, $J=126$,
payoff $(S_T-1)^+$.

In [ ]:
S0, V0 = 1.0, 0.0625
KAPPA, THETA_H, ETA, RHO = 2.0, 0.0625, 0.5, -0.7
T, J, K = 0.5, 126, 1.0
I_train, I_test = 10_000, 20_000
DEPTH = 4          # ordre de la signature lead-lag (Linear BCS / REG)
M_SIG = 4          # ordre de troncature du tenseur de volatilite (Fourier)
dt = T / J

S_train, V_train, dS_train, sigma_train, _ = simulate_heston(
    S0, V0, KAPPA, THETA_H, ETA, RHO, T, J, I_train, seed=42)
S_test, V_test, dS_test, sigma_test, _ = simulate_heston(
    S0, V0, KAPPA, THETA_H, ETA, RHO, T, J, I_test, seed=123)

y_train = compute_payoff_europeen(S_train, K)
y_test = compute_payoff_europeen(S_test, K)

print("E[S_T]        =", S_test[:, -1].mean(), " (cible 1.0, martingale)")
print("prix MC du call =", y_test.mean())

## 1. Linear BCS

Pipeline **identique a la Sec. 5.1** : seules les trajectoires changent. Les mots de
hedging `{tau, S_lag}^* . S_lead` et la regression du P&L empirique sont reutilises
tels quels depuis `linear_hedging`.

In [ ]:
sig_LL_train = signature_transform(build_path_LL(S_train, T), DEPTH)
sig_LL_test = signature_transform(build_path_LL(S_test, T), DEPTH)
valid_idx = hedging_word_indices(DEPTH)

theta_bcs, pnl_bcs = linear_bcs(sig_LL_train, sig_LL_test, valid_idx, y_train, y_test)
mse_bcs, mean_bcs = pnl_stats(pnl_bcs)
print(f"X0 (appris)  = {theta_bcs[0]:.6f}   (prix MC {y_test.mean():.6f})")
print(f"Linear BCS  MSE  = {mse_bcs:.4e}   (cible papier 4.09e-4)")
print(f"Linear BCS  mean = {mean_bcs:.4e}  (cible papier 1.04e-4)")

## 2. Payoff linearise `xi` (eq. 3.8)

Le call n'est pas polynomial : `build_xi` (forme close de la Sec. 5.1) ne s'applique
pas. Comme le payoff ne depend que de $S_T$, les mots puissances du prix $2^{\otimes m}$
suffisent -- on les regresse (`fit_xi_power_words`).

In [ ]:
xi_call = fit_xi_power_words(sig_LL_train, y_train, DEPTH)
print("xi (mot -> coef) :")
for w, c in xi_call.items():
    print(f"  {('()' if w == () else str(w)):12s} -> {c: .6f}")
print(f"\nR^2 de l'approximation du call par xi = {xi_r2(xi_call, sig_LL_train, y_train, DEPTH):.4f}")

## 3. Linear REG (eq. 3.8-3.9)

`E[S^LL]` a l'ordre `2M = 8`, puis forme quadratique shuffle
$J(\theta)=\theta^\top Q\theta - 2b^\top\theta$ sur la base de hedging.
Cellule la plus lente du notebook (~1-2 min).

In [ ]:
HW = hedging_words(DEPTH)
E_sig = expected_signature(S_train, T, 2 * DEPTH)      # ordre 8

Q, b = build_reg_system(E_sig, HW, xi_call)
theta_reg = solve_reg(Q, b)
pnl_reg = reg_pnl(theta_reg, S_test, HW, T, DEPTH, y_test)
mse_reg, mean_reg = pnl_stats(pnl_reg)

print(f"nb mots HW = {len(HW)} | X0 Linear REG = {theta_reg[0]:.6f}")
print(f"Linear REG  MSE  = {mse_reg:.4e}   (cible papier 4.66e-4)")
print(f"Linear REG  mean = {mean_reg:.4e}  (cible papier 6.78e-4)")

## 4. Reference : Fourier avec la vraie fonction caracteristique de Heston

Borne de reference du tableau 6 : on connait la CF de Heston en forme close, donc pas
de signature du tout. C'est la meilleure couverture atteignable.

In [ ]:
X0_fourier_true = compute_X0(kappa=KAPPA, eta=ETA, rho=RHO, T=T, V0=V0, n_laguerre=128)

delta_true = np.zeros((I_test, J))
for j in range(J):
    delta_true[:, j] = compute_delta_vectorized(
        t=j * dt, S_t=S_test[:, j], V_t=V_test[:, j],
        kappa=KAPPA, theta=THETA_H, eta=ETA, rho=RHO,
        T=T, V0=V0, K=K, n_laguerre=128,
    )

pnl_fourier_true = hedging_pnl(X0_fourier_true, delta_true, S_test, y_test)
mse_ft, mean_ft = pnl_stats(pnl_fourier_true)
print(f"X0 Fourier true = {X0_fourier_true:.6f}   (prix MC {y_test.mean():.6f})")
print(f"Fourier true  MSE  = {mse_ft:.4e}   (cible papier 2.33e-4)")
print(f"Fourier true  mean = {mean_ft:.4e}  (cible papier -5.84e-5)")

## 5. Representation signature de la volatilite CIR

$\sqrt{V_t}=\langle\boldsymbol\sigma,\mathbb{S}(\widehat W)_{0,t}\rangle$ avec
$\widehat W_t=(t,W_t)$ et $\boldsymbol\sigma$ **deterministe**, donne par la recurrence
$f_n(v)$ du papier (p.20-21) -- cf. `signatures.volatility_models.compute_sigma_CIR`.

Le test ci-dessous reconstruit $\sqrt{V_t}$ a partir de la seule signature de $\widehat W$
et la compare a la variance simulee. L'erreur **plafonne** vers 1.2e-2 au lieu de tendre
vers 0 : la recurrence tronque les puissances negatives $1/x_t^k$ du terme de derive.
Sans effet notable sur le MSE de couverture (cf. section suivante).

In [ ]:
sigma_dict = compute_sigma_CIR(V0, KAPPA, THETA_H, ETA, M_SIG)
print("sigma[()]   =", sigma_dict[()],    " (attendu sqrt(V0) = 0.25)")
print("sigma[(2,)] =", sigma_dict[(2,)],  " (attendu 0.25)")
print("sigma[(1,)] =", sigma_dict[(1,)],  " (attendu -0.125)")
print()

W_hat_train = build_W_hat(heston_variance_brownian(T, J, I_train, seed=42), T)
for m in range(1, 6):
    sigma_m = compute_sigma_CIR(V0, KAPPA, THETA_H, ETA, m)
    svec = np.array([sigma_m.get(w, 0.0) for w in sig_words(m, 2)])
    vol_sig = sigma_m[()] + signature_transform(W_hat_train, m, fmt=2) @ svec
    err = np.abs(vol_sig - sigma_train[:, 1:])       # stream : t_1..t_J
    print(f"M={m}: mean|vol_sig - sqrt(V)| = {err.mean():.4e}   max = {err.max():.4e}")

## 6. Fourier BCS

On injecte le sigma **exact** dans le Riccati rescale a tenseur constant
(`fourier.signature_constant_vol`). Aucun apprentissage : BCS designe ici le fait que
la volatilite est connue exactement.

In [ ]:
words_sig = generate_words(M_SIG, 2)          # inclut le mot vide en position 0
word_idx_sig = word_to_index(words_sig)
sigma_arr = dict_to_array(sigma_dict, words_sig, word_idx_sig)

f_call = lambda t, u: 1j * u                  # Sec. 3.1, cas call europeen

X0_fourier_bcs = compute_X0_sig_fourier_constant_rescaled(
    RHO, f_call, T, J, M_SIG, words_sig, word_idx_sig, sigma_arr,
    V0=V0, K=K, n_laguerre=32, N_steps=10,
)
print(f"X0 Fourier BCS = {X0_fourier_bcs:.6f}   (prix MC {y_test.mean():.6f})")

W_hat_test = build_W_hat(heston_variance_brownian(T, J, I_test, seed=123), T)
sig_W_tilde = rescale_stream_signature(
    signature_transform(W_hat_test, M_SIG, fmt=2), words_sig, word_idx_sig, dt)

delta_fourier_bcs = compute_delta_sig_fourier_constant_rescaled(
    RHO, S_test, K, f_call, sig_W_tilde, np.maximum(sigma_test, 1e-8),
    T, J, M_SIG, words_sig, word_idx_sig, sigma_arr,
    V0=V0, N_steps=10, n_laguerre=32,
)

pnl_fourier_bcs = hedging_pnl(X0_fourier_bcs, delta_fourier_bcs, S_test, y_test)
mse_fbcs, mean_fbcs = pnl_stats(pnl_fourier_bcs)
print(f"Fourier BCS  MSE  = {mse_fbcs:.4e}   (cible papier 2.33e-4)")
print(f"Fourier BCS  mean = {mean_fbcs:.4e}")

## 7. Calibration de sigma (Algorithme 2) puis Fourier REG

Fourier REG n'a **pas** droit a la forme close du sigma : il ne voit que des moments.
On apparie la signature esperee observee de $\widehat Y=(t,\sqrt V)$ avec sa prediction
theorique $\langle \ell^v, \mathbb{E}[\mathbb{S}(\widehat W)_{0,t}]\rangle$, ou
$\mathbb{E}[\mathbb{S}(\widehat W)_{0,t}]$ est la formule fermee de Fawcett.

Le mot vide (niveau $\sqrt{V_0}$) n'est **pas** calibre : le mot $(2,)$ de la signature
de $\widehat Y$ mesure l'increment $\sqrt{V_t}-\sqrt{V_0}$. Il est restaure avant
l'injection dans le Riccati.

In [ ]:
MBAR = 2                       # ordre des mots de Y a apparier
M_TRUNC = MBAR + M_SIG         # troncature des tenseurs ell^v
words_sigma = sigma_words(M_SIG)                  # mots calibres : SANS le mot vide
t_grid_stream = np.linspace(0, T, J + 1)[1:]      # mode stream : t_1..t_J

E_W_list = E_W_hat_all_times(t_grid_stream, M_TRUNC)   # Fawcett, independant du modele
w2i_Y = word_to_index(sig_words(MBAR, 2))

Y_hat_train = build_Y_hat(sigma_train, T)
Ybar_stream = signature_transform(Y_hat_train, MBAR, fmt=2).mean(axis=0)

loss_fn = make_loss_fn(words_sigma, MBAR, M_TRUNC, E_W_list, Ybar_stream, w2i_Y)

print("Calibration de sigma_hat (Heston)...")
theta_cal, loss_cal = calibrate_sigma(
    loss_fn, sigma_dict, words_sigma, n_restarts=3, steps=100, lr=1e-2, seed=0)
sigma_cal_dict = vec_to_sigma_dict(theta_cal, words_sigma)

print(f"\nperte calibree   = {loss_cal:.6e}")
print(f"perte vrai sigma = {loss_fn(np.array([sigma_dict.get(w, 0.0) for w in words_sigma])):.6e}")
print("\ncoefficients (calibre vs exact) :")
for w in words_sigma[:6]:
    print(f"  {str(w):10s} calibre={sigma_cal_dict[w]: .5f}   exact={sigma_dict.get(w, 0.0): .5f}")

In [ ]:
sigma_cal_arr = dict_to_array(sigma_cal_dict, words_sig, word_idx_sig)
sigma_cal_arr[word_idx_sig[()]] = np.sqrt(V0)     # niveau de base, non calibre

X0_fourier_reg = compute_X0_sig_fourier_constant_rescaled(
    RHO, f_call, T, J, M_SIG, words_sig, word_idx_sig, sigma_cal_arr,
    V0=V0, K=K, n_laguerre=32, N_steps=10,
)

delta_fourier_reg = compute_delta_sig_fourier_constant_rescaled(
    RHO, S_test, K, f_call, sig_W_tilde, np.maximum(sigma_test, 1e-8),
    T, J, M_SIG, words_sig, word_idx_sig, sigma_cal_arr,
    V0=V0, N_steps=10, n_laguerre=32,
)

pnl_fourier_reg = hedging_pnl(X0_fourier_reg, delta_fourier_reg, S_test, y_test)
mse_freg, mean_freg = pnl_stats(pnl_fourier_reg)
print(f"X0 Fourier REG = {X0_fourier_reg:.6f}   (prix MC {y_test.mean():.6f})")
print(f"Fourier REG  MSE  = {mse_freg:.4e}   (cible papier 2.36e-4)")
print(f"Fourier REG  mean = {mean_freg:.4e}  (cible papier -1.05e-3)")

## 8. Densites de P&L hors-echantillon (Figure 9 du papier)

In [ ]:
print(f"{'strategie':22s} {'MSE':>12s} {'mean':>12s}   cible papier (MSE)")
for name, pnl, cible in [
    ("Fourier, true", pnl_fourier_true, "2.33e-4"),
    ("Linear REG", pnl_reg, "4.66e-4"),
    ("Linear BCS", pnl_bcs, "4.09e-4"),
    ("Fourier REG", pnl_fourier_reg, "2.36e-4"),
    ("Fourier BCS", pnl_fourier_bcs, "2.33e-4"),
]:
    print(f"{name:22s} {np.mean(pnl**2):12.4e} {np.mean(pnl):12.4e}   {cible}")

plot_pnl_distribution([
    (pnl_fourier_true, 'Fourier, true', 'tab:blue',   '-'),
    (pnl_reg,          'Linear REG',    'tab:orange', ':'),
    (pnl_bcs,          'Linear BCS',    'tab:green',  '--'),
    (pnl_fourier_reg,  'Fourier REG',   'tab:red',    '-.'),
    (pnl_fourier_bcs,  'Fourier BCS',   'tab:purple', (0, (3, 1, 1, 1))),
], title='European call ATM (Heston)')

---
# Section 5.3 - Call asiatique geometrique sous volatilite a retard

$$d\Sigma_t=\kappa(\theta-\Sigma_t)\,dt+(\eta+\alpha\Sigma_t+\varsigma Z_t)\,dW_t,\qquad
dZ_t=(\Sigma_t-\lambda Z_t)\,dt,\quad Z_0=0$$

$Z_t=\int_0^t e^{-\lambda(t-s)}\Sigma_s\,ds$ est une moyenne exponentiellement ponderee
du passe de $\Sigma$ : c'est elle qui cree la **memoire**. Payoff :
$\big(\exp[\tfrac1T\int_0^T\log S_t\,dt]-K\big)^+$.

**Pourquoi $\boldsymbol\sigma$ est exact ici aussi.** La signature d'un chemin interpole
lineairement est un objet de calcul de **Stratonovich** : on convertit d'abord la derive.
Ensuite, *integrer = concatener a droite* ($\int_0^t Y_s\,d\widehat W^i_s=\langle\ell\otimes e_i,\mathbb{S}_t\rangle$),
ce qui rend le systeme **lineaire a droite** ; on elimine $Z$ puis on inverse :

$$\ell^{Z}=\ell^{\Sigma}\otimes e_1\otimes(\varnothing+\lambda e_1)^{-1},
\qquad \boldsymbol\sigma=\ell^{\Sigma}=\texttt{ell4}\otimes(\varnothing-\boldsymbol\ell)^{-1}.$$

Les deux inverses sont des sommes **finies exactes** dans $T^M$ (nilpotence graduee :
tout mot de $\boldsymbol\ell^{\otimes n}$ a longueur $\ge n$, donc $\mathfrak m^{M+1}=\{0\}$) --
aucune condition de petitesse, aucun rayon de convergence. Cf.
`signatures.volatility_models.build_sigma_DE` et l'avertissement de `tensor_inverse_dict`
sur le mot vide.

> $\Sigma$ n'est **pas** contraint positif ici (min $\approx-0.07$) : les denominateurs en
> $\Sigma_t$ exigent un plancher **signe** (`signed_floor`), pas `np.maximum`.

In [ ]:
SIGMA0, THETA_DE, KAPPA_DE, ETA_DE = 0.25, 0.25, 2.0, -0.12
ALPHA, VARSIGMA, LAM = 1.2, 1.0, -3.0
V0_DE = SIGMA0**2                      # variance initiale (convention du variate BS)

Sigma_train_DE, Z_train_DE, xi_train_DE = simulate_DE(
    SIGMA0, 0.0, KAPPA_DE, THETA_DE, ETA_DE, ALPHA, VARSIGMA, LAM, T, J, I_train, seed=42)
Sigma_test_DE, Z_test_DE, xi_test_DE = simulate_DE(
    SIGMA0, 0.0, KAPPA_DE, THETA_DE, ETA_DE, ALPHA, VARSIGMA, LAM, T, J, I_test, seed=99)

# le prix doit etre correle au VRAI bruit de Sigma : on lui repasse xi
S_DE_train = simulate_S_given_DE(1.0, Sigma_train_DE, xi_train_DE, RHO, T, J, I_train, seed=123)
S_DE_test = simulate_S_given_DE(1.0, Sigma_test_DE, xi_test_DE, RHO, T, J, I_test, seed=456)

y_DE_train = compute_payoff_asian(S_DE_train, K)
y_DE_test = compute_payoff_asian(S_DE_test, K)

print("E[S_T]     =", S_DE_test[:, -1].mean(), " (cible ~1.0)")
print("min Sigma  =", Sigma_test_DE.min(), " (negatif : plancher signe requis)")
print("payoff moyen (prix MC) =", y_DE_test.mean(), "| fraction ITM =", (y_DE_test > 0).mean())

## 1. Representation signature de la volatilite a retard

Meme test que sous Heston : on reconstruit $\Sigma_t$ a partir de la seule signature de
$\widehat W$ et on la compare a la trajectoire simulee.

In [ ]:
sigma_DE = build_sigma_DE(
    sigma0=SIGMA0, kappa=KAPPA_DE, theta=THETA_DE, eta=ETA_DE,
    alpha=ALPHA, varsigma=VARSIGMA, lam=LAM, M=M_SIG,
)

xi_hat_test = build_W_hat(xi_test_DE, T)
sig_xi_stream = signature_transform(xi_hat_test, M_SIG, fmt=2)
svec_DE = np.array([sigma_DE.get(w, 0.0) for w in sig_words(M_SIG, 2)])
Sigma_recon = sigma_DE.get((), 0.0) + sig_xi_stream @ svec_DE

err = np.abs(Sigma_recon - Sigma_test_DE[:, 1:])
print(f"mean abs err   = {err.mean():.6f}")
print(f"median abs err = {np.median(err):.6f}")
print(f"max abs err    = {err.max():.6f}")

## 2. Linear BCS et Linear REG

Le payoff asiatique depend de **toute la trajectoire** : la base des puissances du prix
final ne peut pas le representer. On regresse donc `xi` sur **toutes** les colonnes de
la signature (`fit_xi_full_signature`), ce qui bat le sous-ensemble des mots courts en
train comme en test (cf. docstring de `linear_hedging.payoff_xi`).

In [ ]:
sig_LL_DE_train = signature_transform(build_path_LL(S_DE_train, T), DEPTH)
sig_LL_DE_test = signature_transform(build_path_LL(S_DE_test, T), DEPTH)

theta_DE_bcs, pnl_DE_bcs = linear_bcs(sig_LL_DE_train, sig_LL_DE_test, valid_idx, y_DE_train, y_DE_test)
mse_DE_bcs, mean_DE_bcs = pnl_stats(pnl_DE_bcs)
print(f"Linear BCS (asian)  MSE  = {mse_DE_bcs:.4e}   (cible papier 1.63e-4)")
print(f"Linear BCS (asian)  mean = {mean_DE_bcs:.4e}  (cible papier 2.01e-5)")

In [ ]:
xi_DE = fit_xi_full_signature(sig_LL_DE_train, y_DE_train, DEPTH)
print(f"R^2 train de l'approximation du payoff par xi = {xi_r2(xi_DE, sig_LL_DE_train, y_DE_train, DEPTH):.4f}")

E_sig_DE = expected_signature(S_DE_train, T, 2 * DEPTH)     # ordre 8
Q_DE, b_DE = build_reg_system(E_sig_DE, HW, xi_DE)
theta_DE_reg = solve_reg(Q_DE, b_DE)

pnl_DE_reg = reg_pnl(theta_DE_reg, S_DE_test, HW, T, DEPTH, y_DE_test)
mse_DE_reg, mean_DE_reg = pnl_stats(pnl_DE_reg)
print(f"X0 Linear REG = {theta_DE_reg[0]:.6f}   (prix MC {y_DE_train.mean():.6f})")
print(f"Linear REG (asian)  MSE  = {mse_DE_reg:.4e}   (cible papier 1.74e-4)")
print(f"Linear REG (asian)  mean = {mean_DE_reg:.4e}  (cible papier -1.16e-3)")

## 3. Fourier BCS

Meme machinerie que sous Heston, avec deux changements : $f(t,u)=iu\,(T-t)/T$ (asiatique
au lieu de $iu$), et le variate de controle devient le call asiatique BS. La dependance
de $f$ au temps **calendaire** rend ici visible le passage $t = T-\tau$ du solveur.

In [ ]:
sigma_DE_arr = dict_to_array(sigma_DE, words_sig, word_idx_sig)
f_asian = lambda t, u: 1j * u * (T - t) / T

X0_fourier_bcs_asian = compute_X0_sig_fourier_asian_constant_rescaled(
    RHO, f_asian, T, J, M_SIG, words_sig, word_idx_sig, sigma_DE_arr,
    V0=V0_DE, K=K, n_laguerre=32, N_steps=10,
)
print(f"X0 Fourier BCS (asian) = {X0_fourier_bcs_asian:.6f}   (prix MC {y_DE_test.mean():.6f})")

sig_xi_tilde = rescale_stream_signature(sig_xi_stream, words_sig, word_idx_sig, dt)
sigma_guard = signed_floor(Sigma_test_DE, eps=1e-4)

delta_fourier_bcs_asian = compute_delta_sig_fourier_asian_constant_rescaled(
    RHO, S_DE_test, K, f_asian, sig_xi_tilde, sigma_guard,
    T, J, M_SIG, words_sig, word_idx_sig, sigma_DE_arr,
    V0=V0_DE, N_steps=10, n_laguerre=32,
)

pnl_fourier_bcs_asian = hedging_pnl(X0_fourier_bcs_asian, delta_fourier_bcs_asian, S_DE_test, y_DE_test)
mse_fbcs_a, mean_fbcs_a = pnl_stats(pnl_fourier_bcs_asian)
print(f"Fourier BCS (asian)  MSE  = {mse_fbcs_a:.4e}   (cible papier 3.97e-5)")
print(f"Fourier BCS (asian)  mean = {mean_fbcs_a:.4e}  (cible papier 3.02e-4)")

## 4. Fourier REG

Meme calibration qu'en 5.2, sur les moments de $\widehat Y=(t,\Sigma)$ du modele a retard.
`E_W_list` (Fawcett) ne depend pas du modele : on le reutilise tel quel.

In [ ]:
Y_hat_train_DE = build_Y_hat(Sigma_train_DE, T)
Ybar_stream_DE = signature_transform(Y_hat_train_DE, MBAR, fmt=2).mean(axis=0)

loss_fn_DE = make_loss_fn(words_sigma, MBAR, M_TRUNC, E_W_list, Ybar_stream_DE, w2i_Y)

print("Calibration de sigma_hat (modele a retard)...")
theta_cal_DE, loss_cal_DE = calibrate_sigma(
    loss_fn_DE, sigma_DE, words_sigma, n_restarts=3, steps=100, lr=1e-2, seed=0)
sigma_cal_DE_dict = vec_to_sigma_dict(theta_cal_DE, words_sigma)

print(f"\nperte calibree = {loss_cal_DE:.6e}")
print("coefficients (calibre vs exact) :")
for w in words_sigma[:6]:
    print(f"  {str(w):10s} calibre={sigma_cal_DE_dict[w]: .5f}   exact={sigma_DE.get(w, 0.0): .5f}")

In [ ]:
sigma_cal_DE_arr = dict_to_array(sigma_cal_DE_dict, words_sig, word_idx_sig)
sigma_cal_DE_arr[word_idx_sig[()]] = sigma_DE[()]      # niveau de base, non calibre

X0_fourier_reg_asian = compute_X0_sig_fourier_asian_constant_rescaled(
    RHO, f_asian, T, J, M_SIG, words_sig, word_idx_sig, sigma_cal_DE_arr,
    V0=V0_DE, K=K, n_laguerre=32, N_steps=10,
)
print(f"X0 Fourier REG (asian) = {X0_fourier_reg_asian:.6f}   (prix MC {y_DE_test.mean():.6f})")

delta_fourier_reg_asian = compute_delta_sig_fourier_asian_constant_rescaled(
    RHO, S_DE_test, K, f_asian, sig_xi_tilde, sigma_guard,
    T, J, M_SIG, words_sig, word_idx_sig, sigma_cal_DE_arr,
    V0=V0_DE, N_steps=10, n_laguerre=32,
)

pnl_fourier_reg_asian = hedging_pnl(X0_fourier_reg_asian, delta_fourier_reg_asian, S_DE_test, y_DE_test)
mse_freg_a, mean_freg_a = pnl_stats(pnl_fourier_reg_asian)
print(f"Fourier REG (asian)  MSE  = {mse_freg_a:.4e}   (cible papier 4.00e-5)")
print(f"Fourier REG (asian)  mean = {mean_freg_a:.4e}  (cible papier 6.82e-4)")

## 5. Reference : Black-Scholes naif

On couvre en supposant une volatilite **constante** $\sigma_{BS}=\sqrt{V_0}$, donc en
ignorant entierement la dynamique a retard. Sous BS, $\log G_T$ est gaussien de variance
$\sigma_{BS}^2T/3$ : le prix suit une formule BS a volatilite effective reduite
(Kemna & Vorst 1990). C'est la reference du tableau 7 -- et elle est **meilleure** que
les methodes lineaires, ce que les densites ci-dessous rendent visible.

In [ ]:
X0_naive = compute_call_bs_asian(S=1.0, moy_S=1.0, K=K, t=0.0, T=T, V0=V0_DE)
delta_naive = compute_naive_bs_asian_delta(S_DE_test, K, T, J, V0_DE)
pnl_naive = hedging_pnl(X0_naive, delta_naive, S_DE_test, y_DE_test)

mse_naive, mean_naive = pnl_stats(pnl_naive)
print(f"X0 Naive BS (asian) = {X0_naive:.6f}   (prix MC {y_DE_test.mean():.6f})")
print(f"Naive BS (asian)  MSE  = {mse_naive:.4e}   (cible papier 4.90e-5)")
print(f"Naive BS (asian)  mean = {mean_naive:.4e}  (cible papier 1.91e-4)")

## 6. Densites de P&L hors-echantillon (Figure 11 du papier)

In [ ]:
print(f"{'strategie':22s} {'MSE':>12s} {'mean':>12s}   cible papier (MSE)")
for name, pnl, cible in [
    ("Naive Black-Scholes", pnl_naive, "4.90e-5"),
    ("Linear REG", pnl_DE_reg, "1.74e-4"),
    ("Linear BCS", pnl_DE_bcs, "1.63e-4"),
    ("Fourier REG", pnl_fourier_reg_asian, "4.00e-5"),
    ("Fourier BCS", pnl_fourier_bcs_asian, "3.97e-5"),
]:
    print(f"{name:22s} {np.mean(pnl**2):12.4e} {np.mean(pnl):12.4e}   {cible}")

plot_pnl_distribution([
    (pnl_naive,               'Naive Black-Scholes', 'tab:blue',   '-'),
    (pnl_DE_reg,              'Linear REG',          'tab:orange', ':'),
    (pnl_DE_bcs,              'Linear BCS',          'tab:green',  '--'),
    (pnl_fourier_reg_asian,   'Fourier REG',         'tab:red',    '-.'),
    (pnl_fourier_bcs_asian,   'Fourier BCS',         'tab:purple', (0, (3, 1, 1, 1))),
], title='Geometric Asian call ATM (delayed equation volatility)')